In [ ]:
%matplotlib inline
import sys, os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import butter, filtfilt

sys.path.insert(0, '/Users/yotameviatar/vs_code/Startle')
warnings.filterwarnings('ignore')

from HR import hr_config as config
from HR import hr_demo_processor as processor

plt.rcParams.update({'figure.dpi': 110})
print('Setup complete')

In [23]:
# ── What to show ──────────────────────────────────────────────────────────────
SUBJECT  = 'AB22'          # subject ID
SESSION  = None            # 'eve', 'mor', or None for both

# ── Scipy filter (overlaid on raw; does NOT affect rejection/scoring) ─────────
APPLY_FILTER = False       # flip to True to overlay a filtered version
FILTER_TYPE  = 'lowpass'   # 'lowpass', 'highpass', 'bandpass'
CUTOFF       = [0.3]       # Hz — one value for lp/hp, two for bandpass e.g. [0.05, 0.3]
FILTER_ORDER = 4

# ── Layout ────────────────────────────────────────────────────────────────────
NCOLS     = 6
SAVE_FIGS = False          # False = display only, nothing written to disk

In [24]:
# ── Load cache & run analysis ─────────────────────────────────────────────────
cache_path = os.path.join(config.OUTPUT_DIR, '_cache', 'hr_cache.pkl')
with open(cache_path, 'rb') as f:
    _cache = pickle.load(f)

raw = _cache.get('sessions', _cache.get('trials'))
trials_data, traces = processor.apply_analysis_params(raw, config)

if SUBJECT not in trials_data:
    raise ValueError(f"Subject '{SUBJECT}' not found. Available: {sorted(trials_data.keys())}")

sess_keys = ([SESSION] if SESSION else
             sorted(k for k in trials_data[SUBJECT] if trials_data[SUBJECT][k]))

def _make_ba(sfreq):
    nyq = sfreq / 2.0
    if FILTER_TYPE in ('lowpass', 'highpass'):
        return butter(FILTER_ORDER, CUTOFF[0] / nyq, btype=FILTER_TYPE)
    return butter(FILTER_ORDER, [CUTOFF[0] / nyq, CUTOFF[1] / nyq], btype='bandpass')

def _apply_ba(sig, ba):
    return filtfilt(ba[0], ba[1], sig.astype(float))

for sk in sess_keys:
    trials = trials_data[SUBJECT].get(sk, [])
    n_rej  = sum(t['rejected'] for t in trials)
    reasons = {}
    for t in trials:
        for r in (t.get('rejection_reason') or '').split(','):
            r = r.strip()
            if r: reasons[r] = reasons.get(r, 0) + 1
    reason_str = '  ' + '  '.join(f'{r}×{n}' for r, n in sorted(reasons.items())) if reasons else ''
    filter_state = f'  [filter: {FILTER_TYPE} {CUTOFF} Hz {"ON" if APPLY_FILTER else "OFF"}]'
    print(f"{SUBJECT} {config.SESSION_MAP.get(sk,{}).get('label', sk)}: "
          f"{len(trials)} trials, {n_rej} rejected{reason_str}{filter_state}")

AB22 Evening: 30 trials, 2 rejected  flat_baseline×1  flat_score_window×2  [filter: lowpass [0.3] Hz OFF]
AB22 Morning: 30 trials, 1 rejected  noisy_baseline×1  [filter: lowpass [0.3] Hz OFF]


---
## Rejection Reason Key

| Code | Meaning |
|------|---------|
| `bpm_range` | Baseline mean outside HR_MIN/MAX_BPM — probe off or implausible |
| `noisy_baseline` | Baseline window std is an outlier vs session median |
| `flat_baseline` | Artifact covers >50% of baseline window (probe dropout) |
| `flat_score_window` | Artifact covers >50% of score window (probe dropout) |
| `flat_score_dominant` | Single constant BPM block covers >85% of score window — BPM never updated |
| `score_too_large` | \|Score\| > SCORE_MAX_PCT — implausible % change, likely artifact |
| `atypical_shape` | **Shape outlier:** This trial's BPM trajectory is structurally too different from the session average shape. The tile shows `msd=` (this trial's score) and `cut=` (session threshold) so you can judge the margin. **Tune in config:** `HR_SHAPE_REJECTION_ENABLE` (bool, master on/off) · `HR_SHAPE_SD_THRESHOLD` (float, cutoff = mean + k×std, default 3.0 — raise to rescue marginal trials) · `HR_SHAPE_WINDOW_MIN` / `HR_SHAPE_WINDOW_MAX` (float seconds re t=0, the epoch window fed into the shape matrix). |

> **Note:** Multiple reasons separated by `, ` mean several gates fired on the same trial.

**Blue shading** = baseline window (BASELINE_TMIN → BASELINE_TMAX)  
**Red shading** = score window (SCORE_TMIN → SCORE_TMAX)  
**Red dotted line** = time of max BPM within score window (where the score was measured)

In [ ]:
label_name = {1: 'Neg', 2: 'Neu'}
label_col  = {1: config.NEG_COLOR, 2: config.NEU_COLOR}

def _fmt_reason(rr, t=None):
    parts = [r.strip() for r in rr.split(',') if r.strip()]
    lines = []
    for p in parts:
        if p == 'atypical_shape' and t is not None:
            msd_val = t.get('shape_msd')
            cut_val = t.get('shape_cutoff')
            msd_s = f'{msd_val:.2e}' if msd_val is not None and np.isfinite(float(msd_val)) else '?'
            cut_s = f'{cut_val:.2e}' if cut_val is not None and np.isfinite(float(cut_val)) else '?'
            lines.append(f'atypical_shape\nmsd={msd_s} cut={cut_s}')
        else:
            lines.append(p)
    return '\n'.join(lines)

for sess_key in sess_keys:
    trials = trials_data[SUBJECT].get(sess_key, [])
    if not trials:
        continue
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)
    sfreq = trials[0]['sfreq']
    ba    = _make_ba(sfreq) if APPLY_FILTER else None

    n     = len(trials)
    nrows = max(1, (n + NCOLS - 1) // NCOLS)
    fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 3.0, nrows * 2.4), squeeze=False)
    axes_flat = axes.flatten()

    for i, t in enumerate(trials):
        ax      = axes_flat[i]
        tw      = t['times_wide']
        ep_raw  = t['epoch_wide'].astype(float)
        ep_filt = t.get('epoch_filtered', ep_raw).astype(float)
        rej     = t['rejected']
        lbl     = t.get('label', 0)
        bm      = t.get('baseline_mean')
        sc      = t.get('score')
        rr      = t.get('rejection_reason', '')
        art     = t.get('artifact_mask')
        any_art = t.get('any_artifact', False)

        if rej:
            line_col  = '#95A5A6';  title_col = 'dimgray'
        else:
            line_col  = label_col.get(lbl, '#95A5A6')
            title_col = 'darkorange' if any_art else 'black'

        # Raw BPM staircase (grey)
        ax.step(tw, ep_raw, where='post', color='#BBBBBB', lw=0.8, zorder=2)

        # Optional scipy-filtered overlay
        if ba is not None:
            try:
                ax.plot(tw, _apply_ba(ep_raw, ba), color=line_col, lw=1.2, zorder=4)
            except Exception:
                pass
        else:
            ax.step(tw, ep_filt, where='post', color=line_col, lw=1.0, zorder=3)

        # Artifact spans (orange)
        if art is not None and np.any(art):
            y0, y1 = ep_raw.min() - 0.5, ep_raw.max() + 0.5
            ax.fill_between(tw, y0, y1, where=art,
                            color='darkorange', alpha=0.18, zorder=1)

        # Baseline window (blue, static — same for every trial)
        ax.axvspan(config.BASELINE_TMIN, config.BASELINE_TMAX,
                   alpha=0.10, color='#3498DB', zorder=0)

        # Score window (red, static)
        ax.axvspan(config.SCORE_TMIN, config.SCORE_TMAX,
                   alpha=0.08, color='#E74C3C', zorder=0)

        # Per-trial: mark WHERE the max BPM was found inside the score window
        if not rej:
            score_mask = (tw >= config.SCORE_TMIN) & (tw <= config.SCORE_TMAX)
            if np.any(score_mask):
                ep_score   = ep_filt[score_mask]
                tw_score   = tw[score_mask]
                peak_t     = float(tw_score[np.argmax(ep_score)])
                ax.axvline(peak_t, color='#E74C3C', ls=':', lw=1.0, alpha=0.85)

        # Baseline mean horizontal line
        if bm is not None and not np.isnan(float(bm)):
            ax.axhline(float(bm), color='goldenrod', ls=':', lw=0.9, alpha=0.85)

        ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.45)
        ax.set_xlim(tw[0], tw[-1])
        ax.tick_params(labelsize=5.5)
        ax.spines[['top', 'right']].set_visible(False)

        art_mark = '~' if any_art and not rej else ''
        if rej:
            title = f'T{i+1} REJ\n{_fmt_reason(rr, t) or "?"}'
        else:
            sc_str = f'{sc:.1f}%' if sc is not None and not np.isnan(float(sc)) else '—'
            title  = f'T{i+1} {art_mark}{label_name.get(lbl, "?")} {sc_str}'
        ax.set_title(title, fontsize=6.0, color=title_col, pad=2)

    for ax in axes_flat[n:]:
        ax.set_visible(False)

    filter_note = (f'  |  scipy {FILTER_TYPE} {CUTOFF} Hz  [ON]' if ba
                   else f'  |  filter [{FILTER_TYPE} {CUTOFF} Hz]  OFF')
    fig.suptitle(f'{SUBJECT} — {sess_label} — Individual trials  (N={n}){filter_note}',
                 fontsize=11, fontweight='bold')

    fig.legend(handles=[
        mpatches.Patch(color='#3498DB', alpha=0.5, label='Baseline window'),
        mpatches.Patch(color='#E74C3C', alpha=0.4, label='Score window'),
        plt.Line2D([0],[0], color='#E74C3C', ls=':', label='Max BPM location'),
        plt.Line2D([0],[0], color='goldenrod', ls=':', label='Baseline mean'),
        mpatches.Patch(color='darkorange', alpha=0.4, label='Artifact (interpolated)'),
        plt.Line2D([0],[0], color=config.NEG_COLOR, label='Negative'),
        plt.Line2D([0],[0], color=config.NEU_COLOR, label='Neutral'),
        plt.Line2D([0],[0], color='#95A5A6', label='Rejected'),
    ], fontsize=7, loc='lower center', ncol=8, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'raw_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_trials.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()

---
## View 2 — Full Session Overview
Continuous BPM signal across the entire session.  
**Green lines** = startle onsets where trial passed · **Red lines** = rejected trials  
Y-axis clipped to the physiological range (outlier spikes excluded).

In [ ]:
for sess_key in sess_keys:
    trials = trials_data[SUBJECT].get(sess_key, [])
    if not trials:
        continue
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)

    tr = traces.get(SUBJECT, {}).get(sess_key, {})
    sig = tr.get('signal')
    if sig is None:
        print(f'{sess_key}: no session trace in cache — skipping')
        continue

    sig    = sig.astype(float)
    sfreq  = tr['sfreq']
    t_axis = np.arange(len(sig)) / sfreq
    ba     = _make_ba(sfreq) if APPLY_FILTER else None

    y_lo = max(getattr(config, 'HR_MIN_BPM', 30)  - 10, np.nanpercentile(sig, 1))
    y_hi = min(getattr(config, 'HR_MAX_BPM', 200) + 10, np.nanpercentile(sig, 99))

    fig, ax = plt.subplots(figsize=(20, 3.5))
    ax.step(t_axis, sig, where='post', color='#BBBBBB', lw=0.8, zorder=2, label='raw BPM')
    if ba is not None:
        try:
            ax.plot(t_axis, _apply_ba(sig, ba), color='#2C3E50', lw=0.9, zorder=3,
                    label=f'filtered ({FILTER_TYPE} {CUTOFF} Hz)')
        except Exception:
            pass

    startle_sec = tr['startle_samps_sec']
    n = min(len(startle_sec), len(trials))
    for idx in range(n):
        t_star = float(startle_sec[idx])
        col    = '#27AE60' if not trials[idx].get('rejected') else '#E74C3C'
        ax.axvline(t_star, color=col, lw=1.0, alpha=0.8, zorder=4)

    ax.set_ylim(y_lo, y_hi)
    ax.set_xlabel('Time in session (s)', fontsize=9)
    ax.set_ylabel('BPM', fontsize=9)
    n_rej = sum(1 for t in trials[:n] if t.get('rejected'))
    filter_note = f'  |  filter {FILTER_TYPE} {CUTOFF} Hz  {"ON" if ba else "OFF"}'
    ax.set_title(
        f'{SUBJECT} — {sess_label} — Full session  '
        f'(green=pass, red=reject  {n_rej}/{n} rejected  '
        f'Y {y_lo:.0f}–{y_hi:.0f} BPM){filter_note}',
        fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()

    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'raw_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_session.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()